# Head-to-head on virtual patients: four ways to dose tacrolimus after liver transplant

**Simulated.** Virtual patients come from a published population model of tacrolimus in adult liver transplant recipients:
Bai et al., *Front Pharmacol* 2024, doi 10.3389/fphar.2024.1449535 (CC BY), Table 3: one-compartment oral model, CL/F 37.6 L/h, V/F 1710 L, Ka 0.352 /h,
between-patient variability 66.6% / 126% / 236%, residual error 35.6% proportional + 0.355 µg/L additive.

Dosing twice daily; trough measured each morning; target 8-10 ng/mL (aim 9), as in the CURATE.AI trial (Khong 2025). Days 1-3 fixed at 4 mg twice daily (about the typical-patient dose for a trough near 9),
then each rule sets that day's dose from the troughs so far. Doses in 0.5 mg steps, 0.5-10 mg. Outcome = days 4-14, the same window the trial used (dosing by the method from day 4).

| Rule | What it does |
|---|---|
| fixed | keep 4 mg |
| CURATE.AI linear | dose x 9 / last trough (the trial's "Function 2") |
| CURATE.AI quadratic | fit trough = a + b·dose + c·dose² to all pairs; dose whose fit hits 9 (linear until 3 distinct doses exist) |
| Bayesian (posologyr) | MAP estimate of the patient's own CL, V, Ka from all troughs with the population prior, then the dose predicted to hit 9 |

Scenarios: **calibrated** (main scenario: truth tuned to the real trial's linear arm with held-out checks, see `calibration.ipynb`; variability read as CVs, start 2 mg, 10% residual noise, hospital stay resampled from the 22 real patients; 100 patients), **base** (published noise), **lownoise** (residual error 15% in the truth; the Bayesian prior still assumes the published 35.6%), **drift** (each patient's clearance drifts by a factor of e^±0.5 over the stay; the Bayesian model assumes it is constant).

**Assumptions (not in the source paper):** variability percentages read as log-scale SDs; covariates (weight, bilirubin) left out because their equations are not given in the text; start dose 4 mg; 60 patients per scenario (100 for calibrated); within a scenario every rule sees the same patients and the same noise draws.
Produced by `scripts/sim_tacrolimus.R` (R 4.6.1, posologyr 1.2.8).

In [1]:
import glob
import numpy as np
import pandas as pd
import plotly.graph_objects as go

S = pd.read_csv("../data/tacrolimus_sim.csv")   # output of scripts/sim_tacrolimus.R
S["far"] = (np.maximum(8 - S.trough, S.trough - 10) > 2)
S["inrange"] = S.trough.between(8, 10)
W = S[S.day >= 4]
ORDER = ["fixed", "linear", "quadratic", "bayes"]
per_pat = W.groupby(["scenario", "rule", "id"]).agg(far=("far", "mean"), inrange=("inrange", "mean"),
                                                   mae=("trough", lambda t: np.mean(np.abs(t - 9)))).reset_index()
SUMMARY = per_pat.groupby(["scenario", "rule"])[["far", "inrange", "mae"]].mean().unstack("rule")
print("patients per scenario:", per_pat.groupby("scenario").id.nunique().to_dict())
for metric, label in (("far", "% days far (>2 ng/mL) off target"), ("inrange", "% days in 8-10"), ("mae", "mean |trough - 9| (ng/mL)")):
    t = SUMMARY[metric][ORDER]
    print(f"\n{label}\n" + (t * (100 if metric != "mae" else 1)).round(1).to_string())

patients per scenario: {'base': 60, 'calibrated': 100, 'drift': 60, 'lownoise': 60}

% days far (>2 ng/mL) off target
rule        fixed  linear  quadratic  bayes
scenario                                   
base         75.8    55.5       56.7   50.8
calibrated   77.2    22.5       30.0   14.6
drift        76.1    56.2       64.2   56.5
lownoise     73.6    31.8       42.7   30.5

% days in 8-10
rule        fixed  linear  quadratic  bayes
scenario                                   
base          8.2    13.8       14.2   18.3
calibrated    6.4    34.2       26.9   39.8
drift         9.1    15.8       12.3   14.7
lownoise      8.5    28.3       22.3   31.7

mean |trough - 9| (ng/mL)
rule        fixed  linear  quadratic  bayes
scenario                                   
base          5.7     4.0        4.4    3.5
calibrated    4.9     2.0        2.5    1.7
drift         5.6     3.9        4.7    3.8
lownoise      5.2     2.5        3.5    2.3


In [2]:
rng = np.random.default_rng(0)
def boot_ci(a, b, B=4000):
    a, b = np.asarray(a), np.asarray(b); n = len(a)
    v = [np.mean(a[i] - b[i]) for i in (rng.integers(0, n, n) for _ in range(B))]
    return np.percentile(v, [2.5, 97.5])
PAIRED = {}
for sc in sorted(per_pat.scenario.unique()):
    p = per_pat[per_pat.scenario == sc].pivot(index="id", columns="rule", values="far")
    for a, b in (("bayes", "linear"), ("quadratic", "linear"), ("linear", "fixed")):
        lo, hi = boot_ci(p[a], p[b]); PAIRED[(sc, a, b)] = (float((p[a] - p[b]).mean()), lo, hi)
        print(f"{sc:9s} {a:9s} minus {b:7s}: {100 * (p[a] - p[b]).mean():6.1f} points of far-off days  [{100 * lo:6.1f}, {100 * hi:6.1f}]")

base      bayes     minus linear :   -4.7 points of far-off days  [  -8.5,   -1.1]
base      quadratic minus linear :    1.2 points of far-off days  [  -2.3,    5.0]
base      linear    minus fixed  :  -20.3 points of far-off days  [ -27.4,  -13.0]
calibrated bayes     minus linear :   -7.9 points of far-off days  [ -11.7,   -4.2]
calibrated quadratic minus linear :    7.5 points of far-off days  [   4.7,   10.5]
calibrated linear    minus fixed  :  -54.7 points of far-off days  [ -62.1,  -47.1]


drift     bayes     minus linear :    0.3 points of far-off days  [  -2.9,    3.5]
drift     quadratic minus linear :    8.0 points of far-off days  [   4.1,   12.4]


drift     linear    minus fixed  :  -19.8 points of far-off days  [ -26.7,  -12.9]
lownoise  bayes     minus linear :   -1.4 points of far-off days  [  -3.8,    1.1]
lownoise  quadratic minus linear :   10.9 points of far-off days  [   5.2,   17.1]
lownoise  linear    minus fixed  :  -41.8 points of far-off days  [ -52.0,  -31.8]


In [3]:
fig = go.Figure()
for r in ORDER:
    fig.add_trace(go.Bar(x=SUMMARY["far"].index, y=100 * SUMMARY["far"][r], name=r))
fig.add_hline(y=38.4, line_dash="dot", annotation_text="trial, standard care 38.4%")
fig.add_hline(y=24.2, line_dash="dash", annotation_text="trial, CURATE.AI linear 24.2%")
fig.update_layout(barmode="group", yaxis_title="% days far (>2 ng/mL) off target, days 4-14", height=440,
                  title="Virtual patients: share of days far off target, by rule and scenario")
fig

In [4]:
ex = S[(S.scenario == "base") & (S.id <= 3)]
fig = go.Figure()
for (pid, r), g in ex.groupby(["id", "rule"]):
    fig.add_trace(go.Scatter(x=g.day, y=g.trough, mode="lines+markers", name=f"patient {pid}: {r}", visible=True if pid == 1 else "legendonly"))
fig.add_hrect(y0=8, y1=10, opacity=0.12, line_width=0)
fig.update_layout(xaxis_title="day", yaxis_title="trough (ng/mL)", height=440,
                  title="Example virtual patients (click legend entries to show others)")
fig

## What this says

Main scenario (**calibrated**, realistic on average, see `calibration.ipynb`):

1. **The linear rule lands where the real trial did**: 22.5% of days far off target vs 24.2% in the trial's linear arm. That is partly by construction (the calibration targeted this arm), but it shows the synthetic patients behave like real ones under this rule.
2. **Full Bayesian dosing (posologyr) is clearly better here**: 14.6% far-off days, 7.9 points fewer than the linear rule (95% CI 4.2-11.7). When the noise is realistic, using a population model to separate noise from the patient's real clearance pays off.
3. **CURATE.AI's quadratic variant is worse than its linear variant**: 7.5 points more far-off days (95% CI 4.7-10.5).
4. A fixed dose is far worse (77%), but in this scenario it stays at the low 2 mg start dose, so it is a floor, not a model of usual care (the trial's clinicians reached 38.4%).

Across all four scenarios: the quadratic never beats the linear rule; the Bayesian rule ties or beats it; any feedback beats a fixed dose.

Limits, read before quoting any number:
- Synthetic patients are about 50-60% more varied between patients than the real ones (see calibration); covariates, inter-occasion variability and drug interactions are left out.
- The Bayesian arm uses the published model as its prior, slightly mismatched to the calibrated truth (variability read differently); that is realistic but it is one choice.
- In the base/drift/lownoise scenarios absolute far-off rates are much higher than in the trial because the published residual error (35.6%) is larger than real day-to-day noise; compare rules within a scenario.
- Intervals are bootstrap over virtual patients.